In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
import torch.nn.functional as F
import torchvision.models as models

In [2]:
BATCH_SIZE = 64
NUM_WORKERS = 2
VAL_SPLIT   = 0.2
SEED        = 42

torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [3]:
 # EuroSAT RGB — 27,000 satellite images, 10 land-use classes, 64x64
  # torchvision downloads and extracts it automatically
raw = datasets.EuroSAT(root="./data", download=True)

In [4]:
_loader = DataLoader(datasets.EuroSAT(root="./data",
                      transform=transforms.ToTensor()),
                      batch_size=512, num_workers=NUM_WORKERS)
# mean = torch.zeros(3)
# std  = torch.zeros(3)
# for imgs, _ in _loader:
#     mean += imgs.mean(dim=(0,2,3))
#     std  += imgs.std(dim=(0,2,3))
# mean /= len(_loader)
# std  /= len(_loader)
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std  = [0.229, 0.224, 0.225]

In [5]:
train_transform = transforms.Compose([
      transforms.Resize((224,224)),
      transforms.RandomHorizontalFlip(),
      transforms.RandomVerticalFlip(),
      transforms.RandomRotation(15),
      transforms.RandomCrop(224, padding=28),
      transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
      # transforms.GaussianBlur(kernel_size=3),
      transforms.ToTensor(),
      transforms.Normalize(imagenet_mean, imagenet_std),
  ])

val_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std),   # normalize only, never augment
])

In [6]:
full_train = datasets.EuroSAT(root="./data", transform=train_transform)
full_val   = datasets.EuroSAT(root="./data", transform=val_transform)

n_val   = int(len(full_train) * VAL_SPLIT)
n_train = len(full_train) - n_val
indices = torch.randperm(len(full_train), generator=torch.Generator().manual_seed(SEED))
train_idx, val_idx = indices[:n_train], indices[n_train:]

train_set = torch.utils.data.Subset(full_train, train_idx)
val_set   = torch.utils.data.Subset(full_val,   val_idx)

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS)
val_loader   = DataLoader(val_set,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

In [7]:
model = models.resnet18(weights='IMAGENET1K_V1')
for p in model.parameters():
    p.requires_grad = False
model.fc = nn.Linear(512, 10)

In [8]:
model = model.to(device)

In [9]:
epochs = 40
lr = 0.001
best_acc=0.8637

criterion = nn.CrossEntropyLoss()
optimizer = optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=epochs)
for epoch in range(epochs):
  train_loss=0
  for X,labels in train_loader:
    X,labels = X.to(device), labels.to(device)
    optimizer.zero_grad()
    yp = model(X)

    loss = criterion(yp, labels)
    loss.backward()
    optimizer.step()
    train_loss += loss.item()
  scheduler.step()
  #validation loop

  acc=0
  val_loss=0
  model.eval()
  with torch.no_grad():
    for X,labels in val_loader:
      X,labels = X.to(device), labels.to(device)

      yp = model(X)
      val_loss += criterion(yp,labels).item()
      preds = yp.argmax(dim=1)
      acc+=(preds==labels).sum()
  val_accuracy = acc / len(val_set)
  if val_accuracy > best_acc:
      best_acc = val_accuracy
      torch.save(model.state_dict(), './best_model(build1).pth')
      print(f"  saved new best model with acc {best_acc:.4f}")
  print(f"epoch {epoch+1} | train loss: {train_loss/len(train_loader):.4f} | val loss: {val_loss/len(val_loader):.4f} | val acc: {val_accuracy:.4f}")
  model.train()






epoch 1 | train loss: 0.8691 | val loss: 0.6500 | val acc: 0.7881
epoch 2 | train loss: 0.4926 | val loss: 0.6218 | val acc: 0.7931
epoch 3 | train loss: 0.4390 | val loss: 0.5025 | val acc: 0.8304
epoch 4 | train loss: 0.4064 | val loss: 0.5333 | val acc: 0.8256
epoch 5 | train loss: 0.3847 | val loss: 0.4344 | val acc: 0.8496
epoch 6 | train loss: 0.3716 | val loss: 0.4560 | val acc: 0.8469
epoch 7 | train loss: 0.3567 | val loss: 0.4206 | val acc: 0.8522
epoch 8 | train loss: 0.3536 | val loss: 0.4771 | val acc: 0.8357
  saved new best model with acc 0.8637
epoch 9 | train loss: 0.3517 | val loss: 0.3905 | val acc: 0.8637
epoch 10 | train loss: 0.3398 | val loss: 0.4187 | val acc: 0.8585
epoch 11 | train loss: 0.3382 | val loss: 0.4137 | val acc: 0.8574
epoch 12 | train loss: 0.3367 | val loss: 0.4257 | val acc: 0.8567
epoch 13 | train loss: 0.3343 | val loss: 0.4437 | val acc: 0.8533
epoch 14 | train loss: 0.3291 | val loss: 0.4055 | val acc: 0.8637
  saved new best model with acc 